# SupplyMind AI — Random Forest

Model selection is performed on validation data only.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
# -------------------
# Reload evaluation code
# -------------------

import importlib

import supplymind.features.predictions.ml.evaluation as evaluation

evaluation = importlib.reload(evaluation)

positive_class_probability = evaluation.positive_class_probability
choose_threshold = evaluation.choose_threshold
evaluate_probabilities = evaluation.evaluate_probabilities
BinaryMetrics = evaluation.BinaryMetrics

print("Evaluation module:", evaluation.__file__)
print("BinaryMetrics fields:", BinaryMetrics.__annotations__)

from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_random_forest,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

Evaluation module: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/src/supplymind/features/predictions/ml/evaluation.py
BinaryMetrics fields: {'accuracy': 'float', 'precision': 'float', 'recall': 'float', 'f1': 'float', 'roc_auc': 'float', 'average_precision': 'float', 'balanced_accuracy': 'float', 'specificity': 'float', 'true_negative': 'int', 'false_positive': 'int', 'false_negative': 'int', 'true_positive': 'int', 'false_positive_rate': 'float', 'false_negative_rate': 'float', 'threshold': 'float'}


In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_random_forest()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    [
        "balanced_accuracy",
        "f1",
        "recall",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).head(10)

Selected threshold: 0.38000000000000017


,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
36,0.691120,0.881121,0.537364,0.667590,0.739477,0.835119,0.719194,0.901024,8885,976,6228,7234,0.098976,0.462636,0.56
38,0.690906,0.881141,0.536919,0.667251,0.739477,0.835119,0.719022,0.901126,8886,975,6234,7228,0.098874,0.463081,0.58
40,0.690863,0.881313,0.536696,0.667128,0.739477,0.835119,0.719012,0.901328,8888,973,6237,7225,0.098672,0.463304,0.60
37,0.690906,0.881048,0.536993,0.667282,0.739477,0.835119,0.719009,0.901024,8885,976,6233,7229,0.098976,0.463007,0.57
39,0.690863,0.881220,0.536770,0.667159,0.739477,0.835119,0.718999,0.901227,8887,974,6236,7226,0.098773,0.463230,0.59
41,0.690820,0.881298,0.536622,0.667067,0.739477,0.835119,0.718975,0.901328,8888,973,6238,7224,0.098672,0.463378,0.61
42,0.690777,0.881377,0.536473,0.666975,0.739477,0.835119,0.718951,0.901430,8889,972,6240,7222,0.098570,0.463527,0.62
43,0.690734,0.881362,0.536399,0.666913,0.739477,0.835119,0.718914,0.901430,8889,972,6241,7221,0.098570,0.463601,0.63
35,0.690949,0.880044,0.537884,0.667681,0.739477,0.835119,0.718897,0.899909,8874,987,6221,7241,0.100091,0.462116,0.55
44,0.690606,0.881412,0.536102,0.666697,0.739477,0.835119,0.718816,0.901531,8890,971,6245,7217,0.098469,0.463898,0.64


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.6482442224413669,
 'precision': 0.6747772902539556,
 'recall': 0.7539741494577329,
 'f1': 0.712180746561886,
 'roc_auc': 0.7394773912125235,
 'average_precision': 0.8351188650637533,
 'balanced_accuracy': 0.6289392094008064,
 'specificity': 0.5039042693438799,
 'true_negative': 4969,
 'false_positive': 4892,
 'false_negative': 3312,
 'true_positive': 10150,
 'false_positive_rate': 0.49609573065612006,
 'false_negative_rate': 0.2460258505422671,
 'threshold': 0.38000000000000017}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "random_forest"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)